# Classical NER with Hidden Markov Model (HMM) on WikiANN

This notebook shows a full classical Named Entity Recognition (NER) pipeline using a **supervised Hidden Markov Model (HMM)**.

We will do the following:

1. Load the WikiANN dataset
2. Preprocess the data
3. Train a classical HMM from count statistics
4. Predict tags with the Viterbi algorithm
5. Evaluate the model with precision, recall, F1-score, and accuracy

## Why HMM?

Hidden Markov Model (HMM) is a classical probabilistic model for sequence labeling.

For NER:
- the **observed sequence** is the sentence words
- the **hidden sequence** is the NER labels such as `B-PER`, `I-PER`, `B-LOC`, `O`, etc.

HMM assumes:
- the current label depends on the previous label
- the current word depends on the current label

Summary:
- it is easy to explain
- it is interpretable
- it represents a classical sequence labeling approach

In [1]:
# If needed, install required packages.
# Run this cell only once. If packages are already installed, you can skip it.

%pip install datasets seqeval scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from datasets import load_dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

## Configuration

Here we set the language and some limits.

WikiANN is multilingual.  
For this notebook, we will use:

- `vi` for Vietnamese

You can later change it to:
- `en` for English
- `fr` for French
- etc.

We also use an optional sample limit so the notebook runs faster during testing.

In [3]:
# =========================
# CONFIG
# =========================

LANGUAGE = "en"          # Change to "en" or another WikiANN language if needed
TRAIN_LIMIT = 5000       # Set to None to use full training set
VALID_LIMIT = 1000       # Set to None to use full validation set
TEST_LIMIT = 1000        # Set to None to use full test set

LOWERCASE = True         # Lowercase tokens for more stable emissions
UNK_TOKEN = "<UNK>"
START_TAG = "<START>"
END_TAG = "<END>"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Load the WikiANN dataset

WikiANN already contains:
- `tokens`: the sentence words
- `ner_tags`: the label ids
- label names from the dataset features

Typical labels are:
- `O`
- `B-PER`, `I-PER`
- `B-ORG`, `I-ORG`
- `B-LOC`, `I-LOC`

In [4]:
from pathlib import Path
from datasets import load_dataset, load_from_disk

# =========================
# FIX: force project root
# =========================
PROJECT_ROOT = Path.cwd().parent   # go up from /models → project root

LANGUAGE = "vi"
DATASET_NAME = f"wikiann_{LANGUAGE}"
DATA_DIR = PROJECT_ROOT / "data" / "raw" / DATASET_NAME

# =========================
# LOAD OR DOWNLOAD
# =========================
if DATA_DIR.exists():
    print(f"Loading dataset from local: {DATA_DIR}")
    dataset = load_from_disk(DATA_DIR)
else:
    print("Downloading dataset...")
    dataset = load_dataset("wikiann", LANGUAGE)
    
    print(f"Saving dataset to: {DATA_DIR}")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    dataset.save_to_disk(DATA_DIR)

dataset

Saving dataset to: /Users/kafe/Desktop/Deep Learning_Codes/GroupProject_NER/data/raw/wikiann_vi


Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

In [5]:
label_names = dataset["train"].features["ner_tags"].feature.names
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

## Inspect one sample

In [6]:
sample = dataset["train"][0]
print("Tokens:", sample["tokens"])
print("Tag IDs:", sample["ner_tags"])
print("Tag names:", [label_names[i] for i in sample["ner_tags"]])

Tokens: ['Đồng', 'bằng', 'sông', 'Cửu', 'Long']
Tag IDs: [5, 6, 6, 6, 6]
Tag names: ['B-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'I-LOC']


## Preprocessing

We convert each example into:

- a list of tokens
- a list of tag strings

We also optionally lowercase the tokens.

This makes the data easier to use for a hand-built HMM.

In [7]:
def preprocess_split(hf_split, label_names, limit=None, lowercase=True):
    data = []
    
    n = len(hf_split) if limit is None else min(limit, len(hf_split))
    
    for i in range(n):
        example = hf_split[i]
        tokens = example["tokens"]
        tags = [label_names[tag_id] for tag_id in example["ner_tags"]]
        
        if lowercase:
            tokens = [t.lower() for t in tokens]
        
        # Safety check
        if len(tokens) != len(tags):
            continue
        
        # Skip empty examples
        if len(tokens) == 0:
            continue
        
        data.append((tokens, tags))
    
    return data

train_data = preprocess_split(dataset["train"], label_names, TRAIN_LIMIT, LOWERCASE)
valid_data = preprocess_split(dataset["validation"], label_names, VALID_LIMIT, LOWERCASE)
test_data  = preprocess_split(dataset["test"], label_names, TEST_LIMIT, LOWERCASE)

print("Train size:", len(train_data))
print("Valid size:", len(valid_data))
print("Test size :", len(test_data))

Train size: 5000
Valid size: 1000
Test size : 1000


In [8]:
train_data[0]

(['đồng', 'bằng', 'sông', 'cửu', 'long'],
 ['B-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'I-LOC'])

## Build the vocabulary

Classical HMMs struggle with unseen words.

To handle this, we:
- build a vocabulary from training data
- replace unknown test words with a special token: `<UNK>`

This is important so the model does not completely fail on unseen words.

In [9]:
def build_vocab(data, min_freq=1):
    counter = Counter()
    for tokens, _ in data:
        counter.update(tokens)
    
    vocab = {token for token, freq in counter.items() if freq >= min_freq}
    vocab.add(UNK_TOKEN)
    return vocab, counter

vocab, token_counter = build_vocab(train_data, min_freq=1)

print("Vocabulary size:", len(vocab))
print("Most common tokens:", token_counter.most_common(20))

Vocabulary size: 6743
Most common tokens: [("''", 1965), ('đổi', 1234), ('(', 1023), (')', 1009), ("'", 996), (',', 978), ('.', 432), ('quốc', 329), ('của', 327), ('xã', 251), ('quận', 196), ('-', 162), ('hoàng', 162), ('nam', 162), ('``', 148), ('gia', 139), ('học', 139), ('đại', 137), ('đế', 132), ('và', 125)]


In [10]:
def replace_oov_tokens(data, vocab):
    processed = []
    for tokens, tags in data:
        new_tokens = [t if t in vocab else UNK_TOKEN for t in tokens]
        processed.append((new_tokens, tags))
    return processed

train_data_proc = replace_oov_tokens(train_data, vocab)
valid_data_proc = replace_oov_tokens(valid_data, vocab)
test_data_proc  = replace_oov_tokens(test_data, vocab)

## HMM training idea

A supervised HMM for NER can be estimated from simple counts.

We compute:

1. **Initial probabilities**  
   Probability of the first tag in a sentence

2. **Transition probabilities**  
   Probability of the next tag given the current tag

3. **Emission probabilities**  
   Probability of a word given a tag

To avoid zero probabilities, we use **Laplace smoothing**.

In [11]:
class SupervisedHMMNER:
    def __init__(self, smoothing=1.0, unk_token="<UNK>", start_tag="<START>", end_tag="<END>"):
        self.smoothing = smoothing
        self.unk_token = unk_token
        self.start_tag = start_tag
        self.end_tag = end_tag
        
        self.tags = []
        self.vocab = set()
        
        self.tag_counts = Counter()
        self.initial_counts = Counter()
        self.transition_counts = defaultdict(Counter)
        self.emission_counts = defaultdict(Counter)
        
        self.initial_log_probs = {}
        self.transition_log_probs = defaultdict(dict)
        self.emission_log_probs = defaultdict(dict)
        
        self._fitted = False

    def fit(self, data):
        """
        data: list of (tokens, tags)
        """
        all_tags = set()
        vocab = set()
        
        # Count statistics
        for tokens, tags in data:
            if not tokens or not tags or len(tokens) != len(tags):
                continue
            
            vocab.update(tokens)
            all_tags.update(tags)
            
            self.initial_counts[tags[0]] += 1
            
            for i, (token, tag) in enumerate(zip(tokens, tags)):
                self.tag_counts[tag] += 1
                self.emission_counts[tag][token] += 1
                
                if i < len(tags) - 1:
                    next_tag = tags[i + 1]
                    self.transition_counts[tag][next_tag] += 1
        
        self.tags = sorted(all_tags)
        self.vocab = set(vocab)
        self.vocab.add(self.unk_token)
        
        num_tags = len(self.tags)
        vocab_size = len(self.vocab)
        num_sentences = len(data)
        
        # Initial probabilities with Laplace smoothing
        for tag in self.tags:
            prob = (self.initial_counts[tag] + self.smoothing) / (num_sentences + self.smoothing * num_tags)
            self.initial_log_probs[tag] = math.log(prob)
        
        # Transition probabilities with Laplace smoothing
        for tag_i in self.tags:
            total_out = sum(self.transition_counts[tag_i].values())
            for tag_j in self.tags:
                prob = (self.transition_counts[tag_i][tag_j] + self.smoothing) / (total_out + self.smoothing * num_tags)
                self.transition_log_probs[tag_i][tag_j] = math.log(prob)
        
        # Emission probabilities with Laplace smoothing
        for tag in self.tags:
            total_emit = sum(self.emission_counts[tag].values())
            for token in self.vocab:
                prob = (self.emission_counts[tag][token] + self.smoothing) / (total_emit + self.smoothing * vocab_size)
                self.emission_log_probs[tag][token] = math.log(prob)
        
        self._fitted = True
        return self

    def _get_emission_log_prob(self, tag, token):
        if token not in self.vocab:
            token = self.unk_token
        return self.emission_log_probs[tag].get(token, float("-inf"))

    def predict_one(self, tokens):
        """
        Viterbi decoding for one sentence.
        """
        if not self._fitted:
            raise ValueError("Model must be fitted before prediction.")
        
        if not tokens:
            return []
        
        tokens = [t if t in self.vocab else self.unk_token for t in tokens]
        
        # viterbi[tag][t] = best log score ending in tag at time t
        viterbi = {}
        backpointer = {}
        
        # Initialization
        for tag in self.tags:
            viterbi[tag] = [float("-inf")] * len(tokens)
            backpointer[tag] = [None] * len(tokens)
            viterbi[tag][0] = self.initial_log_probs[tag] + self._get_emission_log_prob(tag, tokens[0])
        
        # Recursion
        for t in range(1, len(tokens)):
            for curr_tag in self.tags:
                best_prev_tag = None
                best_score = float("-inf")
                
                emit_score = self._get_emission_log_prob(curr_tag, tokens[t])
                
                for prev_tag in self.tags:
                    score = (
                        viterbi[prev_tag][t - 1]
                        + self.transition_log_probs[prev_tag][curr_tag]
                        + emit_score
                    )
                    if score > best_score:
                        best_score = score
                        best_prev_tag = prev_tag
                
                viterbi[curr_tag][t] = best_score
                backpointer[curr_tag][t] = best_prev_tag
        
        # Termination
        best_last_tag = None
        best_last_score = float("-inf")
        
        last_t = len(tokens) - 1
        for tag in self.tags:
            if viterbi[tag][last_t] > best_last_score:
                best_last_score = viterbi[tag][last_t]
                best_last_tag = tag
        
        # Backtrack
        best_path = [best_last_tag]
        for t in range(last_t, 0, -1):
            best_path.append(backpointer[best_path[-1]][t])
        
        best_path.reverse()
        return best_path

    def predict(self, data_tokens):
        """
        data_tokens: list of token lists
        """
        return [self.predict_one(tokens) for tokens in data_tokens]

## Train the HMM

In [12]:
hmm_model = SupervisedHMMNER(
    smoothing=1.0,
    unk_token=UNK_TOKEN,
    start_tag=START_TAG,
    end_tag=END_TAG
)

hmm_model.fit(train_data_proc)

print("Number of tags:", len(hmm_model.tags))
print("Tags:", hmm_model.tags)
print("Vocabulary size:", len(hmm_model.vocab))

Number of tags: 7
Tags: ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
Vocabulary size: 6743


In [13]:
## Run prediction on validation and test sets

In [14]:
X_valid = [tokens for tokens, tags in valid_data_proc]
y_valid = [tags for tokens, tags in valid_data_proc]

X_test = [tokens for tokens, tags in test_data_proc]
y_test = [tags for tokens, tags in test_data_proc]

valid_preds = hmm_model.predict(X_valid)
test_preds = hmm_model.predict(X_test)

print("Validation predictions:", len(valid_preds))
print("Test predictions:", len(test_preds))

Validation predictions: 1000
Test predictions: 1000


## Quick sanity check

Let us inspect a few predictions manually.

In [15]:
for i in range(3):
    print(f"Sentence {i+1}")
    print("Tokens      :", X_test[i])
    print("Gold tags   :", y_test[i])
    print("Pred tags   :", test_preds[i])
    print("-" * 80)

Sentence 1
Tokens      : ['hà', 'nội', '&', 'thành', 'phố', 'hồ', 'chí', 'minh']
Gold tags   : ['B-LOC', 'I-LOC', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'I-LOC']
Pred tags   : ['B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG']
--------------------------------------------------------------------------------
Sentence 2
Tokens      : ['chính', 'cha', 'lãm', 'đã', 'trực', 'tiếp', 'gặp', 'nguyễn', 'ngọc', 'loan', 'để', '<UNK>', 'cho', 'ra', '<UNK>', '.']
Gold tags   : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O']
Pred tags   : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O']
--------------------------------------------------------------------------------
Sentence 3
Tokens      : ['đổi', 'ốc', 'móng', 'tay']
Gold tags   : ['O', 'B-LOC', 'I-LOC', 'I-LOC']
Pred tags   : ['O', 'B-LOC', 'I-LOC', 'I-LOC']
-------------------------------------------------------------------------------

## Evaluation

We evaluate with:

- Precision
- Recall
- F1-score
- Accuracy

In [16]:
valid_precision = precision_score(y_valid, valid_preds)
valid_recall = recall_score(y_valid, valid_preds)
valid_f1 = f1_score(y_valid, valid_preds)
valid_accuracy = accuracy_score(y_valid, valid_preds)

test_precision = precision_score(y_test, test_preds)
test_recall = recall_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds)
test_accuracy = accuracy_score(y_test, test_preds)

results_df = pd.DataFrame([
    {
        "split": "validation",
        "precision": valid_precision,
        "recall": valid_recall,
        "f1": valid_f1,
        "accuracy": valid_accuracy,
    },
    {
        "split": "test",
        "precision": test_precision,
        "recall": test_recall,
        "f1": test_f1,
        "accuracy": test_accuracy,
    }
])

results_df

,split,precision,recall,f1,accuracy
0,validation,0.710247,0.543733,0.615935,0.790668
1,test,0.711963,0.552750,0.622335,0.783003


## Detailed test classification report

In [17]:
print(classification_report(y_test, test_preds, digits=4))

              precision    recall  f1-score   support

         LOC     0.7067    0.4746    0.5679       335
         ORG     0.6807    0.5736    0.6226       394
         PER     0.7500    0.6000    0.6667       380

   micro avg     0.7120    0.5528    0.6223      1109
   macro avg     0.7125    0.5494    0.6190      1109
weighted avg     0.7123    0.5528    0.6212      1109



## Error analysis

Now we inspect some examples where the predicted tag sequence is different from the gold tag sequence.

This helps explain where the HMM performs well and where it fails.

In [18]:
def show_errors(tokens_list, gold_list, pred_list, max_examples=10):
    shown = 0
    for tokens, gold, pred in zip(tokens_list, gold_list, pred_list):
        if gold != pred:
            print("TOKENS :", tokens)
            print("GOLD   :", gold)
            print("PRED   :", pred)
            print("-" * 100)
            shown += 1
            if shown >= max_examples:
                break

show_errors(X_test, y_test, test_preds, max_examples=10)

TOKENS : ['hà', 'nội', '&', 'thành', 'phố', 'hồ', 'chí', 'minh']
GOLD   : ['B-LOC', 'I-LOC', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'I-LOC']
PRED   : ['B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG']
----------------------------------------------------------------------------------------------------
TOKENS : ['victoria', 'azarenka', '/', '<UNK>', '<UNK>', "''", '(', 'chung', 'kết', ')']
GOLD   : ['B-PER', 'I-PER', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O']
PRED   : ['B-PER', 'I-PER', 'I-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O']
----------------------------------------------------------------------------------------------------
TOKENS : ['khủng', 'hoảng', 'tài', 'chính', 'hoa', 'kỳ', 'năm', '2007']
GOLD   : ['B-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'I-LOC']
PRED   : ['B-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG', 'I-ORG']
--------------------------------------------------------------------------------------------

## Optional: Compare label frequency in training data

This helps us understand whether some tags are rare.

In [19]:
tag_counter = Counter()
for _, tags in train_data_proc:
    tag_counter.update(tags)

tag_freq_df = pd.DataFrame(tag_counter.items(), columns=["tag", "count"]).sort_values("count", ascending=False)
tag_freq_df

,tag,count
4,O,11881
6,I-ORG,6939
1,I-LOC,4275
3,I-PER,3625
0,B-LOC,1921
2,B-PER,1851
5,B-ORG,1839


## Save results

This cell saves:
- evaluation metrics
- detailed predictions

This is useful for GitHub sharing and later report writing.

In [20]:
from pathlib import Path
import json

# FIX: go to project root
PROJECT_ROOT = Path.cwd().parent

output_dir = PROJECT_ROOT / "reports" / "hmm_wikiann_vi"
output_dir.mkdir(parents=True, exist_ok=True)

# Save metrics
results_df.to_csv(output_dir / "metrics.csv", index=False)

# Save predictions
prediction_records = []
for tokens, gold, pred in zip(X_test, y_test, test_preds):
    prediction_records.append({
        "tokens": tokens,
        "gold_tags": gold,
        "pred_tags": pred
    })

with open(output_dir / "predictions.json", "w", encoding="utf-8") as f:
    json.dump(prediction_records, f, ensure_ascii=False, indent=2)

print(f"Saved results to: {output_dir.resolve()}")

Saved results to: /Users/kafe/Desktop/Deep Learning_Codes/GroupProject_NER/reports/hmm_wikiann_vi


## Conclusion

In this notebook, we implemented a classical HMM-based NER system on WikiANN.

Main steps:
- loaded WikiANN
- converted labels to tag strings
- handled unknown words with `<UNK>`
- trained a supervised HMM from counts
- decoded predictions using Viterbi
- measured precision, recall, F1-score, and accuracy

This classical approach is useful as a baseline, but it has clear limitations:
- it depends heavily on word-tag statistics
- it handles unseen and rare words poorly
- it cannot learn deep contextual representations like BiLSTM or BERT

Still, HMM remains a strong example of a traditional sequence labeling method for NER.